In [2]:
import rassine as r

from astropy.io import fits
from astropy.time import Time
from astropy.timeseries import LombScargle

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from scipy.optimize import curve_fit
import scipy.stats as ss
from sklearn.decomposition import PCA

import re  # find pattern in string
from tqdm import tqdm  # progress bar
import logging  # to avoid printing too many warnings

import tools as t

In [3]:
# define the path of the directory where dace data is saved
path_dace = '../data/dace_sun/'
direc = 'Jun2018/'  # sub directory where the specific data is
# load absorption bands
bands = np.genfromtxt('../data/bands.txt', dtype=None, encoding=None, names=True, delimiter=',')

In [ ]:
# iterate over saved files in direc
# save names (observation dates) of the files
dates_files = []
for entry in os.scandir(path_dace + direc):
    if entry.name != 'spectra.txt' and entry.name != 'CCF':
        dates_files.append(entry.name)

dates_files = sorted(dates_files, key=lambda d: datetime.strptime(d, "%Y-%m-%d"))  # sort
print(dates_files)

['2018-06-01', '2018-06-02', '2018-06-03', '2018-06-04', '2018-06-05', '2018-06-06', '2018-06-07', '2018-06-08', '2018-06-09', '2018-06-10', '2018-06-11', '2018-06-12', '2018-06-13', '2018-06-14', '2018-06-15', '2018-06-16', '2018-06-17', '2018-06-18', '2018-06-19', '2018-06-20', '2018-06-21', '2018-06-22', '2018-06-23', '2018-06-24', '2018-06-25', '2018-06-26', '2018-06-27', '2018-06-28', '2018-06-29', '2018-06-30']


In [ ]:
# make a list of the available files that end by A.fits (s1d)
files_list = []
for d in dates_files:  # iterate over directories
    date_path = path_dace + direc + d + '/'
    files = os.scandir(date_path)  # files in directory
    files = list(files)
    files = [d + '/' + i.name for i in files]  # list of file names
    files_list.extend(files)  # save file names
files_list = [s for s in files_list if s.endswith('A.fits')]  # keep only the files that end by A.fits
files_list = np.array(files_list)
print(f's1d files: {len(files_list)}')

# make a list of the available files that end by A.fits (CCF)
files_ccf_list = []
for d in dates_files:  # iterate over directories
    date_path = path_dace + direc + 'CCF/' + d + '/'
    files = os.scandir(date_path)  # files in directory
    files = list(files)
    files = [d + '/' + i.name for i in files]  # list of file names
    files_ccf_list.extend(files)  # save file names
#files_ccf_list = [s for s in files_ccf_list if s.endswith('A.fits')]  # keep only the files that end by A.fits
files_ccf_list = np.array(files_ccf_list)
print(f'ccf files: {len(files_ccf_list)}')

s1d files: 703
ccf files: 703


In [6]:
# keep only ccf files that have a corresponding s1d file
files_s1d = []
files_ccf = []

for i in files_ccf_list:
    s1d_file = i.replace('CCF', 'S1D')
    if s1d_file in files_list:
        files_s1d.append(s1d_file)
        files_ccf.append(i)

files_s1d = np.array(files_s1d)
files_ccf = np.array(files_ccf)
print(len(files_ccf), len(files_s1d))

602 602


In [9]:
path_file = path_dace + direc + files_s1d[0]
hdul = fits.open(path_file)  # open file

wave = hdul[1].data["wavelength"]
flux = hdul[1].data["flux"]
err  = hdul[1].data["error"]

In [10]:
spec = pd.DataFrame({
    "wave": wave,
    "flux": flux
})

spec.to_csv("sun_spectrum.csv", index=False)

In [ ]:
from rassine.imports.data import IndividualBasicRow
from rassine.imports.preprocess_import import preprocess_import

hdul = fits.open(path_file)

row = IndividualBasicRow(...)

spectrum, imported_row = preprocess_import(
    row=row,
    header=hdul[0].header,
    data=hdul[1].data,
    instrument="HARPN",
    plx_mas=0.0,
    drs_style="new",
)

<class 'rassine.imports.data.IndividualBasicRow'>
{'name': 'str', 'raw_filename': 'str', 'mjd': 'np.float64', 'model': 'np.float64', 'rv_mean': 'np.float64', 'rv_shift': 'np.float64', 'vrad': 'np.float64', 'svrad': 'np.float64', 'drift': 'np.float64'}


In [28]:
help(rassine.rassine.functions)

Help on module rassine.rassine.functions in rassine.rassine:

NAME
    rassine.rassine.functions

FUNCTIONS
    empty_ccd_gap(wave: numpy.ndarray[typing.Any, numpy.dtype[numpy.float64]], flux: numpy.ndarray[typing.Any, numpy.dtype[numpy.float64]], left: Union[float, numpy.float64, NoneType] = None, right: Union[float, numpy.float64, NoneType] = None, extended: float = 30.0) -> numpy.ndarray[typing.Any, numpy.dtype[numpy.float64]]
        Ensure a 0 value in the gap between the ccd of HARPS s1d with extended=30 kms extension
        
        Args:
            wave: Wavelength vector.
            flux: Flux vector.
            left: Wavelength of the left CCF gap.
            right: Wavelength of the right CCF gap.
            extended: Extension of the gap in kms.
        
        Returns:
            Flux values with null values inside the specified gap

DATA
    Float = typing.Union[float, numpy.float64]
    NDArray = numpy.ndarray[typing.Any, numpy.dtype[+_ScalarType_co]]
    Optiona